In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("spam.csv",encoding="latin-1")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [3]:
df.shape

(5572, 5)

In [4]:
df.isnull().sum()

v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64

In [5]:
df=df.drop(columns=['Unnamed: 2','Unnamed: 3','Unnamed: 4'])

In [6]:
df.columns=['label','message']

In [7]:
df['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [8]:
print(df.duplicated().sum())

403


In [9]:
df.drop_duplicates(inplace=True)

## preprocessing

In [10]:
df['message']=df['message'].str.lower()

In [11]:
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

In [12]:
ps=PorterStemmer()
def stemming(text):
    stemmed_words=[]
    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)
df['processed_msg']=df['message'].apply(stemming)

In [13]:
from sklearn.preprocessing import LabelEncoder

In [14]:
le=LabelEncoder()
df['label']=le.fit_transform(df['label'])

In [15]:
y=df['label']

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df["processed_msg"])

In [17]:
df.head()

,label,message,processed_msg
0,0,"go until jurong point, crazy.. available only ...","go until jurong point , crazi .. avail onli in..."
1,0,ok lar... joking wif u oni...,ok lar ... joke wif u oni ...
2,1,free entry in 2 a wkly comp to win fa cup fina...,free entri in 2 a wkli comp to win fa cup fina...
3,0,u dun say so early hor... u c already then say...,u dun say so earli hor ... u c alreadi then sa...
4,0,"nah i don't think he goes to usf, he lives aro...","nah i do n't think he goe to usf , he live aro..."


In [39]:
import pickle

with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tf, f)

In [18]:
# another method for vectorization
from nltk.tokenize import word_tokenize

df["tokens"] = df["message"].apply(word_tokenize)

In [19]:
word_to_idx = {}
idx = 1  # reserve 0 for padding

for tokens in df["tokens"]:
    for word in tokens:
        if word not in word_to_idx:
            word_to_idx[word] = idx
            idx += 1

In [20]:
def text_to_sequence(tokens):

    sequence = []

    for word in tokens:
        sequence.append(
            word_to_idx[word]
        )

    return sequence
df["sequence"] = df["tokens"].apply(text_to_sequence)

In [21]:
max_len = max(len(seq) for seq in df["sequence"])
padded_sequences = []

for seq in df["sequence"]:
    padded_seq = seq + [0] * (max_len - len(seq))
    padded_sequences.append(padded_seq)

In [22]:
df['padded_sequences']=padded_sequences

In [23]:
X2=df['padded_sequences']

In [24]:
print(type(X2))

<class 'pandas.core.series.Series'>


In [25]:
import numpy as np

X2 = np.array(df["padded_sequences"].tolist())

## Dataset and Dataloaders 

In [26]:
from sklearn.model_selection import train_test_split

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [28]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [29]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [30]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [31]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## Building RNN

In [32]:
import torch.nn as nn
import torch.optim as optim

In [33]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()
        
        self.hidden_size=hidden_size
        self.num_layers=num_layers

        #RNN layer
        self.rnn=nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        #fully connected layer
        self.fc=nn.Linear(hidden_size,1)

    def forward(self,x):
        out,_=self.rnn(x)
        out=self.fc(out[:,-1,:])
        return out 

In [34]:
input_size=X_train.shape[1]
model=RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [35]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) 

        outputs = torch.sigmoid(outputs.squeeze()) 

        loss = criterion(outputs, yb) 
        loss.backward() 
        optimizer.step() 

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.37769168615341187
epoch = 2/10 and loss = 0.11793021857738495
epoch = 3/10 and loss = 0.1467331051826477
epoch = 4/10 and loss = 0.06079351156949997
epoch = 5/10 and loss = 0.023897454142570496
epoch = 6/10 and loss = 0.01295514591038227
epoch = 7/10 and loss = 0.014508666470646858
epoch = 8/10 and loss = 0.008307578042149544
epoch = 9/10 and loss = 0.005895901005715132
epoch = 10/10 and loss = 0.027630189433693886


In [36]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 98.45261121856866


In [38]:
torch.save(model.state_dict(), "spam_model.pth")

## model 2

In [37]:
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.2, random_state=42
)

In [38]:
train_set2 = TensorDataset(
    torch.from_numpy(X2_train).long(),
    torch.from_numpy(y2_train.values).float()
)

test_set2 = TensorDataset(
    torch.from_numpy(X2_test).long(),
    torch.from_numpy(y2_test.values).float()
)

In [39]:
train_loader2 = DataLoader(train_set2, shuffle=True, batch_size=64)
test_loader2 = DataLoader(test_set2, shuffle=True, batch_size=64)

In [40]:
class RNN2(nn.Module):
    def __init__(self, vocab_size, embedding_dim=50, hidden_size=128, num_layers=1):
        super().__init__()
        
        self.hidden_size=hidden_size
        self.num_layers=num_layers
        self.vocab_size=vocab_size
        self.embedding_dim=embedding_dim
        
        #embedding layer
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )

        #RNN layer
        self.rnn=nn.RNN(embedding_dim, hidden_size, num_layers, batch_first=True)

        #fully connected layer
        self.fc=nn.Linear(hidden_size,1)

    def forward(self,x):
        x = self.embedding(x)
        out,_=self.rnn(x)
        out=self.fc(out[:,-1,:])
        return out 

In [41]:
vocab_size=len(word_to_idx)
model2=RNN2(vocab_size+1)
criterion2 = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model2.parameters())

In [44]:
epochs = 9

for epoch in range(epochs):
    model2.train()

    for Xb, yb in train_loader2:
        optimizer.zero_grad()
        
        outputs = model2(Xb) 

        outputs = outputs.squeeze()

        loss = criterion2(outputs, yb) 
        loss.backward() 
        optimizer.step() 

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/9 and loss = 0.6447362899780273
epoch = 2/9 and loss = 0.382968008518219
epoch = 3/9 and loss = 0.43232405185699463
epoch = 4/9 and loss = 0.33302366733551025
epoch = 5/9 and loss = 0.3347058892250061
epoch = 6/9 and loss = 0.3831184506416321
epoch = 7/9 and loss = 0.3838004469871521
epoch = 8/9 and loss = 0.5507466793060303
epoch = 9/9 and loss = 0.43326011300086975


In [45]:
model2.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader2:

        outputs = model2(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.97678916827853
